In [2]:
import pandas as pd
from collections import defaultdict
import pickle
from wrappers.sarima_wrapper import SARIMA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*force_all_finite.*",
    category=FutureWarning,
    module="sklearn"
)

In [3]:
with open('../data/datasets/data_cleaned.pkl', 'rb') as f:
    states_dfs = pickle.load(f)

In [ ]:
states= [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", 
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", 
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", 
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", 
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
]
horizon = 12
target = 'residential_electricity_price'

283
295
12


In [ ]:
sarima_metrics = {'state': [], 'mape': [], 'model_order': []}
sarima_models = {}
for state in states:
    df = states_dfs[state]
    model = SARIMA(state=state,
               horizon=12, 
               seasonal=True,
               max_order=25,
               max_q=20,
               max_p=4)
    train_data = df[target].iloc[:-horizon]
    test_data = df[target].iloc[-horizon:]
    model.fit(train_data, trace=False)
    preds = model.predict(test_data)
    model.plot_forecast()
    mape = mean_absolute_percentage_error(preds, test_data)
    order = model.model.order
    sarima_metrics['state'].append(state)
    sarima_metrics['mape'].append(mape)
    sarima_metrics['model_order'].append(order)
    sarima_models[state] = model

sarima_metrics = pd.DataFrame(sarima_metrics)

In [ ]:
import os
if not os.path.isdir('metrics'):
    os.makedirs('metrics')
with open('metrics/sarima_metrics.pkl', 'wb') as f:
    pickle.dump(sarima_metrics, f)
f.close()

In [ ]:
sarima_metrics.sort_values(by='mape', ascending=False)